# Ia 型超新星とハッブル図でハッブル定数を求める

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/coxless/astro-sandbox/blob/main/distance/sn_ia_hubble.ipynb)

**Ia 型超新星** は、白色矮星が限界の質量に達して爆発する現象です。
爆発の仕組みがほぼ同じなので、最大光度もほぼそろっています。さらに、光度曲線の幅(減光の速さ)と色で補正すると、明るさのばらつきは約 0.1 mag(距離にして約 5%)まで小さくなります。
銀河 1 つ分ほどの明るさになるので、数十億光年先でも見えます。

遠くの銀河ほど速く遠ざかっている、というのが **ハッブル=ルメートルの法則** です。

$$
v = H_0\, d
$$

$H_0$(ハッブル定数)は宇宙の膨張の速さを表し、宇宙の年齢や大きさの基準になります。

このノートブックでやること:

1. Pantheon+ の Ia 型超新星のデータを読み込む
2. セファイドで距離が分かっている銀河の超新星を使って、超新星の絶対等級 $M_B$ を較正する
3. 遠くの超新星にその $M_B$ を当てはめて距離を出し、ハッブル定数 $H_0$ を求める
4. 赤方偏移 2 くらいまでのハッブル図を描き、宇宙の加速膨張を見る

前のノートブック(セファイド)から続く **距離はしご**:

> 視差 → セファイドの周期−光度関係 → (セファイドのいる銀河で)超新星の明るさを較正 → 遠方の超新星 → $H_0$

## 使うデータ

- **Pantheon+**(Scolnic et al. 2022, Brout et al. 2022)と SH0ES(Riess et al. 2022)が論文と一緒に GitHub で公開しているデータ:
  https://github.com/PantheonPlusSH0ES/DataRelease
  - リポジトリにライセンスの明記はありません。研究用に公開されたデータなので、使うときは上の論文を引用します。

## 0. 準備

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.cosmology import FlatLambdaCDM

DATA_URL = (
    "https://raw.githubusercontent.com/PantheonPlusSH0ES/DataRelease/main/"
    "Pantheon%2B_Data/4_DISTANCES_AND_COVAR/Pantheon%2BSH0ES.dat"
)
C_KMS = 299_792.458  # 光速 [km/s]

## 1. データを読み込む

1 行が「1 つの超新星を 1 つの観測プロジェクトで測った結果」です。主な列:

| 列 | 意味 |
| --- | --- |
| `CID` | 超新星の名前 |
| `zHD` | 赤方偏移(宇宙背景放射の静止系に直し、銀河の固有運動を補正したもの) |
| `m_b_corr` | 光度曲線の幅と色で補正した見かけの等級 |
| `m_b_corr_err_DIAG` | その誤差 |
| `IS_CALIBRATOR` | 1 なら、セファイドで距離が分かっている銀河の超新星(較正用) |
| `CEPH_DIST` | 較正用超新星の母銀河の、セファイドによる距離指数 |
| `USED_IN_SH0ES_HF` | 1 なら、SH0ES がハッブル定数の計算に使った超新星(0.023 < z < 0.15) |

In [ ]:
raw = pd.read_csv(DATA_URL, sep=r"\s+")
print(f"{len(raw)} 行, 超新星 {raw['CID'].nunique()} 個")
raw[["CID", "zHD", "m_b_corr", "m_b_corr_err_DIAG", "IS_CALIBRATOR", "CEPH_DIST", "USED_IN_SH0ES_HF"]].head()

同じ超新星を複数のプロジェクトが観測していると、行が重複します。
ここでは簡単のため、超新星ごとに平均をとって 1 行にまとめます(誤差は小さくなりすぎないように平均値をそのまま使います)。

In [ ]:
sn = raw.groupby("CID").agg(
    zHD=("zHD", "mean"),
    m_b=("m_b_corr", "mean"),
    m_b_err=("m_b_corr_err_DIAG", "mean"),
    is_cal=("IS_CALIBRATOR", "max"),
    ceph_mu=("CEPH_DIST", "mean"),
    hubble_flow=("USED_IN_SH0ES_HF", "max"),
).reset_index()
print(f"較正用: {(sn['is_cal'] == 1).sum()} 個, ハッブル流: {(sn['hubble_flow'] == 1).sum()} 個")

## 2. まずはざっくり: 見かけの等級と赤方偏移

超新星の本来の明るさがそろっているなら、見かけの等級 $m$ が暗いほど遠いはずです。
近い範囲では $v \approx cz$ なので、$m$ と $\log_{10}(cz)$ は傾き 5 の直線に乗るはずです($m = 5\log_{10} d + \text{定数}$ だから)。

In [ ]:
low = sn[(sn["zHD"] > 0.01) & (sn["zHD"] < 0.15)]
cz = C_KMS * low["zHD"]

plt.figure(figsize=(7, 5))
plt.scatter(np.log10(cz), low["m_b"], s=5)
xx = np.linspace(np.log10(cz).min(), np.log10(cz).max(), 10)
offset = np.median(low["m_b"] - 5 * np.log10(cz))
plt.plot(xx, 5 * xx + offset, c="tab:red", label="slope = 5")
plt.xlabel("log10 cz [km/s]")
plt.ylabel("m_B (corrected) [mag]")
plt.gca().invert_yaxis()
plt.legend()
plt.show()

きれいに傾き 5 の直線に乗ります。ただし、このままでは $H_0$ は決まりません。
**超新星の絶対等級 $M_B$** が分からないと、見かけの等級を距離に直せないからです。

## 3. 超新星の絶対等級をセファイドで較正する

近くの銀河で、セファイドで距離指数 $\mu_{\mathrm{Ceph}}$ が分かっていて、しかも Ia 型超新星が出たものがあります。
その超新星については

$$
M_B = m_B - \mu_{\mathrm{Ceph}}
$$

で絶対等級が直接求まります。

In [ ]:
cal = sn[sn["is_cal"] == 1].copy()
cal["M_B"] = cal["m_b"] - cal["ceph_mu"]

w = 1 / cal["m_b_err"] ** 2
M_B = np.sum(w * cal["M_B"]) / np.sum(w)
M_B_err = cal["M_B"].std() / np.sqrt(len(cal))  # ばらつきから見積もった平均値の誤差
print(f"較正に使った超新星: {len(cal)} 個")
print(f"M_B = {M_B:.3f} ± {M_B_err:.3f}  (ばらつき {cal['M_B'].std():.2f} mag)")

plt.figure(figsize=(7, 4))
plt.errorbar(cal["ceph_mu"], cal["M_B"], yerr=cal["m_b_err"], fmt="o", ms=4)
plt.axhline(M_B, c="tab:red", label=f"mean M_B = {M_B:.2f}")
plt.xlabel("Cepheid distance modulus of host galaxy [mag]")
plt.ylabel("M_B [mag]")
plt.gca().invert_yaxis()
plt.legend()
plt.show()

母銀河の距離(横軸)によらず、$M_B \approx -19.25$ 前後にそろっています。これが「Ia 型超新星は標準光源」ということです。

## 4. ハッブル定数を求める

ハッブル流(銀河の固有運動の影響が小さく、宇宙の膨張に素直に乗っている距離:0.023 < z < 0.15)の超新星について、

1. 距離指数 $\mu = m_B - M_B$ から光度距離 $d_L$ を出す
2. 赤方偏移から「$H_0$ をかけた光度距離」$H_0\,d_L(z)$ を計算する
3. その比が $H_0$

$z$ が小さいときの光度距離は、減速パラメータ $q_0$ と躍度 $j_0$ を使って次のように書けます(Riess et al. 2016, 2022 と同じ形)。

$$
H_0\, d_L(z) \approx cz \left[ 1 + \frac{1 - q_0}{2} z - \frac{1 - q_0 - 3q_0^2 + j_0}{6} z^2 \right]
$$

$q_0 = -0.55$, $j_0 = 1$ を使います(加速膨張している宇宙のおおよその値)。$z < 0.15$ ではこの選び方の影響は小さいです。

In [ ]:
Q0, J0 = -0.55, 1.0

hf = sn[sn["hubble_flow"] == 1].copy()
z = hf["zHD"]
hf["H0_dL"] = C_KMS * z * (1 + (1 - Q0) / 2 * z - (1 - Q0 - 3 * Q0 ** 2 + J0) / 6 * z ** 2)  # [km/s]
hf["mu"] = hf["m_b"] - M_B
hf["dL_mpc"] = 10 ** ((hf["mu"] - 25) / 5)                 # 距離指数 → Mpc
hf["H0"] = hf["H0_dL"] / hf["dL_mpc"]

# 等級の誤差で重みをつけて log10 H0 を平均する
w = 1 / hf["m_b_err"] ** 2
H0 = 10 ** (np.sum(w * np.log10(hf["H0"])) / np.sum(w))

# 誤差の目安: ハッブル流の超新星の選び方(ブートストラップ)+ M_B の較正誤差
rng = np.random.default_rng(0)
logH, ww = np.log10(hf["H0"]).to_numpy(), w.to_numpy()
boot = []
for _ in range(2000):
    i = rng.integers(0, len(hf), len(hf))
    boot.append(10 ** (np.sum(ww[i] * logH[i]) / np.sum(ww[i])))
err_hf = np.std(boot)
err_cal = H0 * np.log(10) / 5 * M_B_err
H0_err = np.hypot(err_hf, err_cal)

print(f"ハッブル流の超新星: {len(hf)} 個")
print(f"H0 = {H0:.1f} ± {H0_err:.1f} km/s/Mpc")
print(f"  (ハッブル流の統計 ±{err_hf:.1f}, M_B の較正 ±{err_cal:.1f})")
print(f"宇宙の年齢の目安 1/H0 = {9778 / H0:.0f} 億年")

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(hf["dL_mpc"], C_KMS * hf["zHD"], s=5, label="Pantheon+ (Hubble flow)")
dd = np.linspace(0, hf["dL_mpc"].max(), 100)
plt.plot(dd, H0 * dd, c="tab:red", label=f"v = H0 d, H0 = {H0:.1f}")
plt.xlabel("luminosity distance d_L [Mpc]")
plt.ylabel("cz [km/s]")
plt.legend()
plt.show()

SH0ES の結果は $H_0 = 73.0 \pm 1.0$ km/s/Mpc(Riess et al. 2022)です。

一方、宇宙マイクロ波背景放射の観測(Planck)から ΛCDM モデルを仮定して出した値は $H_0 = 67.4 \pm 0.5$ km/s/Mpc で、両者は 5σ 程度ずれています。
これが **ハッブル・テンション** と呼ばれる、現在の宇宙論の大きな未解決問題です。

> 注: このノートブックの $M_B$ の較正には、SH0ES が求めたセファイド距離(`CEPH_DIST`)をそのまま使っています。
> 前のノートブックの LMC 距離から自分で全部つなぐには、SH0ES のセファイドの測光データが必要になります(発展課題)。

## 5. 遠くまでのハッブル図と加速膨張

$z \gtrsim 0.1$ になると、距離と赤方偏移の関係は宇宙の膨張の歴史によって変わります。
いくつかの宇宙モデルの予言と、超新星の距離指数を比べます。

- **ΛCDM**: 物質 30%、ダークエネルギー 70%($\Omega_m = 0.3$)
- **物質だけ**: ダークエネルギーなし($\Omega_m = 1$)— 膨張は減速する

In [ ]:
all_sn = sn[sn["zHD"] > 0.01].copy()
all_sn["mu"] = all_sn["m_b"] - M_B

models = {
    "LCDM (Om=0.3, OL=0.7)": FlatLambdaCDM(H0=H0, Om0=0.3),
    "matter only (Om=1)": FlatLambdaCDM(H0=H0, Om0=1.0),
}
zz = np.logspace(-2, np.log10(all_sn["zHD"].max()), 200)
ref = models["LCDM (Om=0.3, OL=0.7)"]

fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
axes[0].errorbar(all_sn["zHD"], all_sn["mu"], yerr=all_sn["m_b_err"], fmt=".", ms=2, alpha=0.3, c="gray")
axes[1].errorbar(all_sn["zHD"], all_sn["mu"] - ref.distmod(all_sn["zHD"].to_numpy()).value,
                 yerr=all_sn["m_b_err"], fmt=".", ms=2, alpha=0.3, c="gray")
for name, cosmo in models.items():
    axes[0].plot(zz, cosmo.distmod(zz).value, label=name)
    axes[1].plot(zz, cosmo.distmod(zz).value - ref.distmod(zz).value, label=name)

axes[0].set_ylabel("distance modulus mu [mag]")
axes[0].legend()
axes[1].set_ylabel("mu - mu(LCDM) [mag]")
axes[1].set_xlabel("redshift z")
axes[1].set_xscale("log")
axes[1].set_ylim(-1, 1)
plt.tight_layout()
plt.show()

遠くの超新星は、ダークエネルギーのない宇宙で予想されるより暗く(=遠くに)見えます。
これが 1998 年に見つかった **宇宙の加速膨張** の証拠です(2011 年ノーベル物理学賞)。

## 発展メモ

- ハッブル流の範囲(`zHD` の下限・上限)を変えると $H_0$ がどう変わるか
- $q_0$ を 0.5(物質だけの宇宙)にすると $H_0$ はどれくらい変わるか
- 本格的には、`Pantheon+SH0ES_STAT+SYS.cov` の共分散行列を使って誤差の相関まで入れて当てはめる

## 参考・クレジット

- Scolnic et al. (2022), *The Pantheon+ Analysis: The Full Data Set and Light-curve Release*, ApJ 938, 113
- Brout et al. (2022), *The Pantheon+ Analysis: Cosmological Constraints*, ApJ 938, 110
- Riess et al. (2022), *A Comprehensive Measurement of the Local Value of the Hubble Constant with 1 km/s/Mpc Uncertainty from the Hubble Space Telescope and the SH0ES Team*, ApJL 934, L7
- Riess et al. (2016), *A 2.4% Determination of the Local Value of the Hubble Constant*, ApJ 826, 56
- Planck Collaboration (2020), *Planck 2018 results. VI. Cosmological parameters*, A&A 641, A6